In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Ml_projects\Accident_Prediction\data\traffic_accidents.csv")
df.head(5)

,crash_date,traffic_control_device,weather_condition,lighting_condition,first_crash_type,trafficway_type,alignment,roadway_surface_cond,road_defect,crash_type,...,most_severe_injury,injuries_total,injuries_fatal,injuries_incapacitating,injuries_non_incapacitating,injuries_reported_not_evident,injuries_no_indication,crash_hour,crash_day_of_week,crash_month
0,07/29/2023 01:00:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,TURNING,NOT DIVIDED,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,13,7,7
1,08/13/2023 12:11:00 AM,TRAFFIC SIGNAL,CLEAR,"DARKNESS, LIGHTED ROAD",TURNING,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,2.0,0,1,8
2,12/09/2021 10:30:00 AM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,DRY,NO DEFECTS,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,10,5,12
3,08/09/2023 07:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,ANGLE,FOUR WAY,STRAIGHT AND LEVEL,DRY,NO DEFECTS,INJURY AND / OR TOW DUE TO CRASH,...,NONINCAPACITATING INJURY,5.0,0.0,0.0,5.0,0.0,0.0,19,4,8
4,08/19/2023 02:55:00 PM,TRAFFIC SIGNAL,CLEAR,DAYLIGHT,REAR END,T-INTERSECTION,STRAIGHT AND LEVEL,UNKNOWN,UNKNOWN,NO INJURY / DRIVE AWAY,...,NO INDICATION OF INJURY,0.0,0.0,0.0,0.0,0.0,3.0,14,7,8


**Remove Duplicates**

In [3]:
print("duplicated_Records:-",df.duplicated().sum())

duplicated_Records:- 31


In [4]:
df=df.drop_duplicates()

**Standardize Columns Names**

In [5]:
df.columns=df.columns.str.lower().str.replace('-','_').str.replace(' ','_').str.strip()

**Fixing The crash_date dtype**

In [6]:
df['crash_date']=pd.to_datetime(df['crash_date'],format='%m/%d/%Y %I:%M:%S %p')

In [7]:
mapping={
    'OVER $1,500':2,
    '$501 - $1,500':1,
    '$500 OR LESS':0
}
df['damage']=df['damage'].map(mapping)

Feature engineering

In [8]:
Numerical_Features=df.select_dtypes(['int','float'])
Categorical_Features=df.select_dtypes(['object'])

In [9]:
df['crash_year']=df['crash_date'].dt.year

In [10]:
df['total_injuries'] = df['injuries_fatal'] + df['injuries_incapacitating'] + df['injuries_non_incapacitating'] 
+ df['injuries_reported_not_evident'] + df['injuries_no_indication']

0         3.0
1         2.0
2         3.0
3         0.0
4         3.0
         ... 
209301    2.0
209302    2.0
209303    0.0
209304    1.0
209305    2.0
Length: 209275, dtype: float64

In [11]:
# Create a feature for crash severity
df['crash_severity'] = np.where(df['injuries_fatal'] > 0, 'Fatal',np.where(df['injuries_incapacitating'] > 0, 'Incapacitating',
np.where(df['injuries_non_incapacitating'] > 0, 'Non-Incapacitating','No Injury')))

In [12]:
df[['total_injuries', 'crash_severity']].head()

,total_injuries,crash_severity
0,0.0,No Injury
1,0.0,No Injury
2,0.0,No Injury
3,5.0,Non-Incapacitating
4,0.0,No Injury


In [13]:
df.drop(columns=['injuries_fatal','injuries_incapacitating','injuries_non_incapacitating',
                 'injuries_reported_not_evident','injuries_no_indication','crash_date'],inplace=True)

**Encoding**

In [14]:
Encoded_features=pd.get_dummies(Categorical_Features,drop_first=True).replace({True:1,False:0}).astype(int)

C:\Users\P\AppData\Local\Temp\ipykernel_21508\2303742838.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Encoded_features=pd.get_dummies(Categorical_Features,drop_first=True).replace({True:1,False:0}).astype(int)


In [15]:
df.drop(columns=Categorical_Features,inplace=True)

In [16]:
df=pd.concat([Encoded_features,df],axis=1)

**Store Process Data**

In [21]:
df.to_csv("C:\Ml_projects\Accident_Prediction\data\clean_traffic_accidents.csv", index=False)

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\P\AppData\Local\Temp\ipykernel_21508\2509316701.py:1: SyntaxWarning: invalid escape sequence '\M'
  df.to_csv("C:\Ml_projects\Accident_Prediction\data\clean_traffic_accidents.csv", index=False)
